In [ ]:
# Start by importing pytorch
!pip install torch

import torch
import torch.nn as nn

import random
device = torch.device('cuda')
from transformers import GPT2LMHeadModel, GPT2Tokenizer
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

In [ ]:
# Add EOS token for proper probability calculation
def probability(sentence):
  input_ids = gpt2_tokenizer.encode(sentence, return_tensors="pt", add_special_tokens=True)
  input_ids = input_ids.to(device)
  with torch.no_grad():
      outputs = gpt2_model(input_ids, labels=input_ids)
      logits = outputs.logits
  # Shift logits and labels to align them for next-token prediction
  shift_logits = logits[:, :-1, :].contiguous()
  shift_labels = input_ids[:, 1:].contiguous()

  # Calculate log-probabilities
  log_probs = nn.functional.log_softmax(shift_logits, dim=-1)
  # Gather the log-probabilities of the actual next tokens
  token_log_probs = torch.gather(log_probs, -1, shift_labels.unsqueeze(-1)).squeeze(-1)
  sentence_log_prob = token_log_probs.sum().item()
  sentence_prob = torch.exp(torch.tensor(sentence_log_prob)).item()
  return sentence_prob

In [ ]:
def gpt2_transitive(word):
  sentence_1 = "I " + word + "."
  sentence_2 = "I " + word + " it"

  return (probability(sentence_1) - probability(sentence_2))

In [ ]:
def gpt2_who_vs_what(word):
  who_sentence = "Who " + word
  what_sentence = "What " + word
  return (probability(who_sentence) - probability(what_sentence))

In [ ]:
def gpt2_who_vs_what_transitive(word):
  who_sentence = "Who " + word + " it"
  what_sentence = "What " + word + " it"
  return (probability(who_sentence) - probability(what_sentence))

In [ ]:
def gpt2_animate(word):
  sentence_animate = "I " + word + " them"
  sentence_inanimate = "I " + word + " it"
  return (probability(sentence_animate) - probability(sentence_inanimate))

In [ ]:
intransitive_verbs = ['agreed', 'appeared', 'arrived', 'awakened', 'advanced', 'achieved', 'acted', 'adapted', 'applauded', 'aged', 'ascended', 'assembled', 'assisted', 'approached', 'arose', 'apologized', 'accumulated', 'attended', 'aspired', 'breathed', 'belonged', 'became', 'bloomed', 'blushed', 'barked', 'bowed', 'began', 'buzzed', 'bounced', 'blinked', 'basked', 'came', 'collapsed', 'cheered', 'complained', 'cried', 'changed', 'chilled', 'chimed', 'conversed', 'conformed', 'commuted', 'ceased', 'consented', 'crackled', 'converged', 'coexisted', 'cowered', 'crawled', 'danced', 'drifted', 'descended', 'disappeared', 'dined', 'dawdled', 'developed', 'departed', 'dove', 'differed', 'decreased', 'delighted', 'dawned', 'emerged', 'existed', 'escaped', 'ended', 'expanded', 'evolved', 'exploded', 'engaged', 'erupted', 'entered', 'embarked', 'echoed', 'eased', 'enlisted', 'faded', 'fell', 'floated', 'flew', 'fled', 'flourished', 'frowned', 'fumbled', 'froze', 'functioned', 'fluttered', 'faltered', 'feasted', 'glowed', 'glistened', 'glided', 'grinned', 'groaned', 'giggled', 'galloped', 'grew', 'gasped', 'glimmered', 'gushed', 'happened', 'healed', 'hovered', 'hurried', 'hesitated', 'hiccuped', 'heaved', 'hoped', 'hustled', 'halted', 'hibernated', 'hitchhiked', 'harmonized', 'improved', 'intervened', 'increased', 'insisted', 'invested', 'inflated', 'illustrated', 'interfered', 'inquired', 'imagined', 'interrupted', 'jogged', 'jumped', 'jittered', 'joked', 'journied', 'jived', 'jiggled', 'jeered', 'jested', 'journaled', 'kneeled', 'knocked', 'laughed', 'leaped', 'listened', 'lingered', 'lasted', 'lied', 'lived', 'lounged', 'landed', 'lurked', 'led', 'learned', 'limped', 'marched', 'migrated', 'melted', 'moved', 'multiplied', 'mellowed', 'meddled', 'misbehaved', 'nodded', 'napped', 'nested', 'occurred', 'overcame', 'overreacted', 'obeyed', 'objected', 'partied', 'paused', 'proceeded', 'played', 'performed', 'persisted', 'prevailed', 'peaked', 'protested', 'progressed', 'posed', 'pounced', 'pouted', 'prayed', 'preened', 'quivered', 'quieted', 'quickened', 'quit', 'reclined', 'reflected', 'rested', 'rejoiced', 'recovered', 'roared', 'relaxed', 'relented', 'rose', 'rolled' 'ran', 'rushed', 'sailed', 'smiled', 'slept', 'stood', 'sneezed', 'swam', 'shined', 'stopped', 'shivered', 'shrunk', 'survived', 'succeeded', 'stumbled', 'screamed', 'shouted', 'sighed', 'sat', 'skipped', 'slid', 'snarled', 'soaked', 'spun', 'spat', 'sprinted', 'squeaked', 'traveled', 'trembled', 'thrived', 'turned', 'taught', 'tiptoed', 'tired', 'twirled', 'twisted', 'unloaded', 'vanished', 'volunteered', 'voted', 'waded', 'walked', 'wandered', 'waved', 'whirled', 'wiggled', 'worked', 'whistled', 'worked', 'winked', 'withdrew', 'yawned', 'yelled', 'yielded', 'yodeled', 'yelped']
intransitive_edited = []
for verb in intransitive_verbs:
  intransitive_edited.append([verb, gpt2_transitive(verb)])
intransitive_edited.sort(key=lambda x : x[1])
intransitive_edited = intransitive_edited[149:]
len(intransitive_edited)

100

In [ ]:
intransitive_final = []
for verb in intransitive_edited:
  if (gpt2_who_vs_what(verb[0]) > 0):
    intransitive_final.append(verb[0])
print(intransitive_final)

['quivered', 'faded', 'rested', 'proceeded', 'reflected', 'led', 'belonged', 'progressed', 'differed', 'yielded', 'glowed', 'interfered', 'began', 'whistled', 'voted', 'arose', 'fled', 'buzzed', 'volunteered', 'delighted', 'landed', 'yawned', 'acted', 'evolved', 'erupted', 'emerged', 'withdrew', 'squeaked', 'attended', 'partied', 'winked', 'rejoiced', 'leaped', 'shouted', 'flourished', 'roared', 'objected', 'intervened', 'insisted', 'cheered', 'lingered', 'stood', 'interrupted', 'trembled', 'apologized', 'recovered', 'escaped', 'fell', 'prevailed', 'faltered', 'sneezed', 'froze', 'listened', 'consented', 'jumped', 'appeared', 'exploded', 'relented', 'protested', 'complained', 'vanished', 'arrived', 'pounced', 'screamed', 'obeyed', 'yelped', 'gasped', 'tired', 'moved', 'persisted', 'paused', 'relaxed', 'hesitated', 'survived', 'giggled', 'collapsed', 'came', 'blinked', 'shivered', 'rose', 'grinned', 'cried', 'blushed', 'disappeared', 'quit', 'frowned', 'lied', 'groaned', 'sighed', 'stop

In [ ]:
 animate = ['fired', 'annoyed', 'punished', 'respected', 'devastated', 'addressed', 'sheltered', 'adored', 'summoned', 'scared', 'chased', 'suspected', 'congratulated', 'recruited', 'repelled', 'challenged', 'befriended', 'compensated', 'greeted', 'arrested', 'consulted', 'promised', 'enraged', 'deceived', 'mocked', 'trusted', 'praised', 'embraced', 'reassured', 'forgave', 'disappointed', 'hugged', 'reminded', 'thanked', 'begged', 'informed', 'employed', 'embarrassed', 'encouraged', 'invited', 'notified', 'threatened', 'adopted', 'protected', 'chastised', 'vilified', 'shocked', 'blamed', 'lectured', 'accompanied', 'loved', 'visited', 'watched', 'rescued', 'sponsored', 'confused', 'welcomed', 'frightened', 'pleased', 'escorted', 'astonished', 'bored', 'stunned', 'rewarded', 'questioned', 'healed', 'admired', 'persuaded', 'hired', 'called', 'interrupted', 'offended', 'instructed', 'complimented', 'startled', 'paid', 'disturbed', 'intimidated', 'enlisted', 'warned', 'surprised', 'empowered', 'alerted', 'promoted', 'appointed', 'mentored', 'prosecuted', 'helped', 'judged', 'guided', 'blessed', 'nominated', 'honored', 'followed', 'married', 'robbed', 'interviewed', 'commanded', 'destroyed', 'asked', 'bewildered', 'kissed', 'joined', 'flattered', 'convinced', 'bothered', 'charged', 'impressed', 'irritated', 'entertained', 'amazed', 'comforted', 'carried', 'served', 'funded', 'imitated', 'aggravated', 'scolded', 'enlightened', 'educated', 'cautioned', 'assured', 'fed', 'contradicted', 'advised', 'told']
inanimate = ['collected', 'specified', 'opened', 'identified', 'maintained', 'split', 'dismissed', 'rejected', 'cancelled', 'published', 'proposed', 'tried', 'wore', 'prepared', 'presented', 'refilled', 'made', 'folded', 'listed', 'widened', 'tasted', 'spread', 'fixed', 'forgot', 'improved', 'loaded', 'cited', 'recorded', 'celebrated', 'tied', 'ordered', 'delivered', 'witnessed', 'mentioned', 'summarized', 'built', 'stored', 'cleaned', 'conveyed', 'delayed', 'enclosed', 'edited', 'printed', 'recited', 'risked', 'understood', 'spent', 'felt', 'enjoyed', 'lightened', 'debated', 'hammered', 'typed', 'got', 'spilled', 'reported', 'remembered', 'shared', 'imagined', 'cut', 'outlined', 'chopped', 'heated', 'denied', 'questioned', 'defined', 'dried', 'broadcasted', 'pulled', 'formulated', 'drank', 'highlighted', 'composed', 'copied', 'emptied', 'wrote', 'sewed', 'designed', 'threw', 'boiled', 'grasped', 'poured', 'ignored', 'managed', 'bought', 'passed', 'polished', 'solved', 'escaped', 'justified', 'filled', 'planted', 'pressed', 'connected', 'stated', 'tightened', 'experienced', 'melted', 'measured', 'handled', 'sorted', 'wanted', 'sold', 'packed', 'smelled', 'changed', 'admitted', 'purchased', 'dusted', 'locked', 'illustrated', 'acknowledged', 'updated', 'peeled', 'explained', 'burned', 'noticed', 'posted', 'lifted', 'studied', 'noted', 'assembled', 'grabbed', 'analyzed', 'ironed', 'rewrote', 'communicated', 'caught', 'realized', 'dropped', 'dictated', 'articulated', 'scanned', 'clarified', 'enforced']
animate_edited = []
inanimate_edited = []
for verb in animate:
  animate_edited.append([verb, gpt2_transitive(verb)])
animate_edited.sort(key=lambda x : x[1])
for verb in inanimate:
  inanimate_edited.append([verb, gpt2_transitive(verb)])
inanimate_edited.sort(key=lambda x : x[1])


In [ ]:
animate_final = []
inanimate_final = []
for verb in animate_edited:
  if (gpt2_animate(verb[0]) > 0):
    animate_final.append(verb[0])
for verb in inanimate_edited:
  if (gpt2_animate(verb[0]) < 0):
    inanimate_final.append(verb[0])
